## 1. All Asset Prices Over Time

In [2]:
fig, ax = plt.subplots(figsize=(14, 7))
for asset in prices.columns:
    sid = meta.loc[asset, 'sector_id']
    ax.plot(prices.index, prices[asset], color=SECTOR_COLORS[sid], alpha=0.6, linewidth=0.9)
handles = [Line2D([0],[0], color=SECTOR_COLORS[s], linewidth=2, label=SECTOR_NAMES[s]) for s in sorted(SECTOR_NAMES)]
ax.legend(handles=handles, loc='upper left')
ax.set_title('All Asset Prices Over Time (colored by sector)')
ax.set_xlabel('Tick'); ax.set_ylabel('Price'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

NameError: name 'plt' is not defined

## 2. Price Paths by Sector (faceted)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharey=False)
for sid, ax in enumerate(axes):
    assets = meta[meta['sector_id'] == sid].index.tolist()
    for asset in assets:
        ax.plot(prices.index, prices[asset], label=asset, linewidth=1)
    ax.set_title(f'Sector {sid}'); ax.set_xlabel('Tick'); ax.set_ylabel('Price')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
fig.suptitle('Price Paths by Sector', fontsize=14)
plt.tight_layout(); plt.show()

## 3. Average Cumulative Return (%) by Sector

In [ ]:
returns_pct = prices - 100.0
fig, ax = plt.subplots(figsize=(14, 6))
for sid in sorted(SECTOR_NAMES):
    assets = meta[meta['sector_id'] == sid].index.tolist()
    sector_avg = returns_pct[assets].mean(axis=1)
    ax.plot(prices.index, sector_avg, color=SECTOR_COLORS[sid], linewidth=2, label=SECTOR_NAMES[sid])
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Average Cumulative Return (%) by Sector')
ax.set_xlabel('Tick'); ax.set_ylabel('Return (%)'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Final Price Distribution (sorted, colored by sector)

In [ ]:
final_prices = prices.iloc[-1].sort_values()
colors = [SECTOR_COLORS[meta.loc[a, 'sector_id']] for a in final_prices.index]
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(final_prices.index, final_prices.values, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(100, color='black', linestyle='--', linewidth=1, label='Start price (100)')
handles = [Patch(color=SECTOR_COLORS[s], label=SECTOR_NAMES[s]) for s in sorted(SECTOR_NAMES)]
ax.legend(handles=handles, loc='upper left')
ax.set_title(f'Final Prices at Tick {prices.index[-1]} (sorted ascending)')
ax.set_xlabel('Asset'); ax.set_ylabel('Price'); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Rolling Volatility by Sector

In [ ]:
tick_returns = prices.pct_change() * 100
WINDOW = 20
fig, ax = plt.subplots(figsize=(14, 6))
for sid in sorted(SECTOR_NAMES):
    assets = meta[meta['sector_id'] == sid].index.tolist()
    sector_vol = tick_returns[assets].std(axis=1).rolling(WINDOW).mean()
    ax.plot(prices.index, sector_vol, color=SECTOR_COLORS[sid], linewidth=2, label=SECTOR_NAMES[sid])
ax.set_title(f'Rolling {WINDOW}-tick Cross-Sectional Volatility by Sector')
ax.set_xlabel('Tick'); ax.set_ylabel('Avg Std of Tick Returns (%)'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Return Correlation Heatmap (assets sorted by sector)

In [ ]:
corr = tick_returns.corr()
asset_order = meta.sort_values('sector_id').index.tolist()
corr_sorted = corr.loc[asset_order, asset_order]
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr_sorted, ax=ax, cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7}, linewidths=0.5, square=True)
boundaries = meta.sort_values('sector_id').groupby('sector_id').size().cumsum().tolist()
for b in boundaries[:-1]:
    ax.axhline(b, color='black', linewidth=1.5)
    ax.axvline(b, color='black', linewidth=1.5)
ax.set_title('Asset Return Correlation Matrix (sorted by sector)')
plt.tight_layout(); plt.show()

## 7. Spread & Borrow Cost vs. Realized Volatility / Final Return

In [ ]:
realized_vol = tick_returns.std()
final_ret    = prices.iloc[-1] - 100.0
summary = meta.copy()
summary['realized_vol'] = realized_vol
summary['final_return'] = final_ret

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
for sid in sorted(SECTOR_NAMES):
    sub = summary[summary['sector_id'] == sid]
    ax.scatter(sub['spread_bps'], sub['realized_vol'], color=SECTOR_COLORS[sid], s=80, label=SECTOR_NAMES[sid], zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row.name, (row['spread_bps'], row['realized_vol']), fontsize=7, textcoords='offset points', xytext=(4,2))
ax.set_xlabel('Spread (bps)'); ax.set_ylabel('Realized Volatility (% per tick)')
ax.set_title('Spread vs. Realized Volatility'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
for sid in sorted(SECTOR_NAMES):
    sub = summary[summary['sector_id'] == sid]
    ax.scatter(sub['borrow_bps_annual'], sub['final_return'], color=SECTOR_COLORS[sid], s=80, label=SECTOR_NAMES[sid], zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(row.name, (row['borrow_bps_annual'], row['final_return']), fontsize=7, textcoords='offset points', xytext=(4,2))
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Borrow Cost (bps annual)'); ax.set_ylabel('Final Return (%)')
ax.set_title('Borrow Cost vs. Final Return'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()